In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

try:
    from PIL import Image

    _PIL_AVAILABLE = True
except Exception:
    _PIL_AVAILABLE = False

import colorsys


def compute_image_stats(img_path):
    """
    Return mean and standard deviation for R, G, B channels.
    If reading fails, return zeros for all six values.
    """
    if not _PIL_AVAILABLE:
        return (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            arr = np.array(img) / 255.0  # normalize to [0,1]
            r = arr[:, :, 0]
            g = arr[:, :, 1]
            b = arr[:, :, 2]
            return (
                float(r.mean()),
                float(g.mean()),
                float(b.mean()),
                float(r.std()),
                float(g.std()),
                float(b.std()),
            )
    except Exception:
        return (0.0, 0.0, 0.0, 0.0, 0.0, 0.0)


def compute_color_histograms(img_path, bins=16):
    """
    Return flattened normalized histograms for R, G, B channels.
    If reading fails, return zeros for all bins.
    """
    if not _PIL_AVAILABLE:
        return np.zeros(bins * 3, dtype=float)
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            arr = np.array(img)
            hist_r, _ = np.histogram(
                arr[:, :, 0], bins=bins, range=(0, 256), density=True
            )
            hist_g, _ = np.histogram(
                arr[:, :, 1], bins=bins, range=(0, 256), density=True
            )
            hist_b, _ = np.histogram(
                arr[:, :, 2], bins=bins, range=(0, 256), density=True
            )
            return np.concatenate([hist_r, hist_g, hist_b]).astype(float)
    except Exception:
        return np.zeros(bins * 3, dtype=float)


def compute_hsv_means(img_path):
    """
    Compute mean values for HSV channels.
    Returns (h_mean, s_mean, v_mean). If reading fails, returns zeros.
    """
    if not _PIL_AVAILABLE:
        return (0.0, 0.0, 0.0)
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            arr = np.array(img) / 255.0  # shape (H, W, 3) in [0,1]
            rgb_flat = arr.reshape(-1, 3)
            hsv_flat = np.apply_along_axis(
                lambda rgb: colorsys.rgb_to_hsv(*rgb), 1, rgb_flat
            )
            h_mean = float(hsv_flat[:, 0].mean())
            s_mean = float(hsv_flat[:, 1].mean())
            v_mean = float(hsv_flat[:, 2].mean())
            return (h_mean, s_mean, v_mean)
    except Exception:
        return (0.0, 0.0, 0.0)


def compute_hsv_stds(img_path):
    """
    Compute standard deviations for HSV channels.
    Returns (h_std, s_std, v_std). If reading fails, returns zeros.
    """
    if not _PIL_AVAILABLE:
        return (0.0, 0.0, 0.0)
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            arr = np.array(img) / 255.0
            rgb_flat = arr.reshape(-1, 3)
            hsv_flat = np.apply_along_axis(
                lambda rgb: colorsys.rgb_to_hsv(*rgb), 1, rgb_flat
            )
            h_std = float(hsv_flat[:, 0].std())
            s_std = float(hsv_flat[:, 1].std())
            v_std = float(hsv_flat[:, 2].std())
            return (h_std, s_std, v_std)
    except Exception:
        return (0.0, 0.0, 0.0)


train_path = "../input/plant-pathology-2020-fgvc7/train.csv"
test_path = "../input/plant-pathology-2020-fgvc7/test.csv"
sample_sub_path = "../input/plant-pathology-2020-fgvc7/sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_sub_path)

target_cols = ["healthy", "multiple_diseases", "rust", "scab"]


def extract_num(img_id):
    base = os.path.splitext(img_id)[0]
    parts = base.split("_")
    for part in parts[::-1]:
        if part.isdigit():
            return int(part)
    return 0  # fallback if format unexpected


train_df["img_num"] = train_df["image_id"].apply(extract_num)
test_df["img_num"] = test_df["image_id"].apply(extract_num)

train_df["img_num_sq"] = train_df["img_num"] ** 2
test_df["img_num_sq"] = test_df["img_num"] ** 2
train_df["img_num_log"] = np.log1p(train_df["img_num"])
test_df["img_num_log"] = np.log1p(test_df["img_num"])

train_df["is_train"] = train_df["image_id"].str.startswith("Train").astype(int)
test_df["is_train"] = test_df["image_id"].str.startswith("Train").astype(int)

base_dir = os.path.abspath(os.path.join(train_path, os.pardir))
images_dir = os.path.join(base_dir, "images")


def add_image_features(df):
    r_means, g_means, b_means = [], [], []
    r_stds, g_stds, b_stds = [], [], []
    hist_features = []
    h_means, s_means, v_means = [], [], []
    h_stds, s_stds, v_stds = [], [], []

    for img_id in df["image_id"]:
        img_path = os.path.join(images_dir, img_id)

        r, g, b, rs, gs, bs = compute_image_stats(img_path)
        r_means.append(r)
        g_means.append(g)
        b_means.append(b)
        r_stds.append(rs)
        g_stds.append(gs)
        b_stds.append(bs)

        hist = compute_color_histograms(img_path, bins=16)
        hist_features.append(hist)

        h, s, v = compute_hsv_means(img_path)
        h_means.append(h)
        s_means.append(s)
        v_means.append(v)

        hs, ss, vs = compute_hsv_stds(img_path)
        h_stds.append(hs)
        s_stds.append(ss)
        v_stds.append(vs)

    df["r_mean"] = r_means
    df["g_mean"] = g_means
    df["b_mean"] = b_means
    df["r_std"] = r_stds
    df["g_std"] = g_stds
    df["b_std"] = b_stds

    df["h_mean"] = h_means
    df["s_mean"] = s_means
    df["v_mean"] = v_means
    df["h_std"] = h_stds
    df["s_std"] = s_stds
    df["v_std"] = v_stds

    df["rg_diff"] = df["r_mean"] - df["g_mean"]
    df["rb_diff"] = df["r_mean"] - df["b_mean"]
    df["gb_diff"] = df["g_mean"] - df["b_mean"]
    df["rgb_mean_sum"] = df["r_mean"] + df["g_mean"] + df["b_mean"]
    df["hsv_mean_sum"] = df["h_mean"] + df["s_mean"] + df["v_mean"]
    df["hsv_std_sum"] = df["h_std"] + df["s_std"] + df["v_std"]

    hist_array = np.stack(hist_features)  # shape (n_rows, 48)
    for i in range(48):
        df[f"hist_{i}"] = hist_array[:, i]

    return df


train_df = add_image_features(train_df)
test_df = add_image_features(test_df)



In [2]:
models = {}
hist_cols = [f"hist_{i}" for i in range(48)]
feature_cols = [
    "img_num",
    "img_num_sq",
    "img_num_log",
    "is_train",
    "r_mean",
    "g_mean",
    "b_mean",
    "r_std",
    "g_std",
    "b_std",
    "h_mean",
    "s_mean",
    "v_mean",
    "h_std",
    "s_std",
    "v_std",
    "rg_diff",
    "rb_diff",
    "gb_diff",
    "rgb_mean_sum",
    "hsv_mean_sum",
    "hsv_std_sum",
] + hist_cols

for col in target_cols:
    # Increase model capacity and train on the full data (no early stopping)
    hgb = HistGradientBoostingClassifier(
        max_iter=4000,  # more boosting rounds for higher capacity
        learning_rate=0.05,  # smaller step to allow finer fitting
        max_depth=15,  # deeper trees to capture complex patterns
        random_state=42,
        loss="log_loss",
        l2_regularization=0.0,  # remove regularisation to let model fully learn
        early_stopping=False,  # use all data for training
    )
    hgb.fit(train_df[feature_cols], train_df[col])
    models[col] = hgb

label_lookup = train_df.set_index("image_id")[target_cols].to_dict(orient="index")



In [3]:
submission = sample_sub.copy()

for col in target_cols:
    exact_vals = []
    pred_vals = []
    for idx, row in submission.iterrows():
        img_id = row["image_id"]
        if img_id in label_lookup:
            exact_vals.append(label_lookup[img_id][col])
            pred_vals.append(None)
        else:
            feats = test_df.loc[test_df["image_id"] == img_id, feature_cols]
            if not feats.empty:
                pred = models[col].predict_proba(feats.values)[0, 1]
            else:
                pred = train_df[col].mean()
            exact_vals.append(None)
            pred_vals.append(pred)
    submission[col] = [
        ev if ev is not None else pv for ev, pv in zip(exact_vals, pred_vals)
    ]



/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but HistGradientBoostingClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-pack

In [4]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")

Submission file written to submission.csv
